In [16]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "FutureHendrix2023-"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Load all records
df = pd.DataFrame.from_records(db.read({}))

# Drop MongoDB _id to avoid DataTable ObjectId issues
if "_id" in df.columns:
    df.drop(columns=['_id'], inplace=True)

print(df.columns)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare’s logo (safe loader)
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = None
if os.path.exists(image_filename):
    encoded_image = base64.b64encode(open(image_filename, 'rb').read())
else:
    print(
        f"Logo file not found: {image_filename} "
        f"(put it in the same folder as the notebook or update the filename)"
    )

app.layout = html.Div([
    # Optional (better): show text when logo missing
    html.Div([
        html.Img(
            src=('data:image/png;base64,{}'.format(encoded_image.decode()) if encoded_image else ''),
            style={'width': '200px'}
        ),
        html.Div(
            "LOGO MISSING: put GraziosoSalvareLogo.png next to this notebook",
            style={'color': 'red', 'fontWeight': 'bold', 'marginTop': '6px'}
        ) if not encoded_image else html.Div()
    ]),

    html.Div("Gavin Elliott - CS 340 Project Two", style={'fontWeight': 'bold'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),

    html.Div([
        html.Label("Rescue Type Filter"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset (All)', 'value': 'Reset'},
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster/Tracking', 'value': 'Disaster'}
            ],
            value='Reset',
            inline=True
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[],
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '100px', 'width': '150px', 'maxWidth': '250px'}
    ),

    html.Br(),
    html.Hr(),

    # side-by-side chart + map
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id', className='col s12 m6')
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

WATER_BREEDS = [
    "Labrador Retriever Mix",
    "Chesapeake Bay Retriever",
    "Newfoundland"
]

MOUNTAIN_BREEDS = [
    "German Shepherd",
    "Alaskan Malamute",
    "Old English Sheepdog",
    "Siberian Husky",
    "Rottweiler"
]

DISASTER_BREEDS = [
    "Doberman Pinscher",
    "German Shepherd",
    "Golden Retriever",
    "Bloodhound",
    "Rottweiler"
]

def base_dog_query():
    return {
        "animal_type": "Dog",
        "age_upon_outcome_in_weeks": {"$lte": 104},
        "sex_upon_outcome": {"$in": ["Intact Male", "Intact Female"]}
    }

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    # Default: show all records
    query = {}

    if filter_type == 'Water':
        query = base_dog_query()
        query["breed"] = {"$in": WATER_BREEDS}

    elif filter_type == 'Mountain':
        query = base_dog_query()
        query["breed"] = {"$in": MOUNTAIN_BREEDS}

    elif filter_type == 'Disaster':
        query = base_dog_query()
        query["breed"] = {"$in": DISASTER_BREEDS}

    dff = pd.DataFrame.from_records(db.read(query))
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')

@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        return [html.Div("No data to display")]

    dff = pd.DataFrame(viewData)

    if "breed" not in dff.columns:
        return [html.Div("No 'breed' column found for chart")]

    fig = px.pie(dff, names='breed', title='Breed Distribution (Filtered)')
    return [dcc.Graph(figure=fig)]

# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# Update the geo-location chart for the selected data entry
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):

    if viewData is None:
        return [html.Div("No data available for map.")]

    # Step 4: show first row by default so the map isn't blank on load
    if index is None:
        index = [0]

    dff = pd.DataFrame.from_dict(viewData)
    row = index[0]

    lat_col = "location_lat"
    lon_col = "location_long"

    if lat_col not in dff.columns or lon_col not in dff.columns:
        return [html.Div(f"Missing map columns: {lat_col} / {lon_col}")]

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.loc[row, lat_col], dff.loc[row, lon_col]],
                    children=[
                        dl.Tooltip(dff.loc[row, "breed"] if "breed" in dff.columns else "Animal"),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.loc[row, "name"] if "name" in dff.columns else "Unknown")
                        ])
                    ]
                )
            ]
        )
    ]

# Run app and display result in jupyterlab mode
app.run_server()

Index(['animal_id', 'animal_type', 'name', 'breed', 'color',
       'age_upon_outcome', 'outcome_type', 'rec_num', 'date_of_birth',
       'datetime', 'monthyear', 'outcome_subtype', 'sex_upon_outcome',
       'location_lat', 'location_long', 'age_upon_outcome_in_weeks'],
      dtype='object')
Dash app running on https://manilanavy-dietgallop-3000.codio.io/proxy/8050/
